# 01 — Data Understanding

> **Insiders Loyalty Program** · Customer Segmentation Project

---

Following Géron's end-to-end ML checklist: **"Get the data, explore it, understand it."** (Ch. 2)

This notebook covers:
1. Load the raw transaction data
2. Inspect schema, shape, and data types
3. Analyse missing values
4. Key statistics and business metrics
5. Transactions over time
6. Data quality summary

In [ ]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Project paths
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from insiders_loyalty_program.config import load_config
from insiders_loyalty_program.data import read_csv, normalize_columns

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('tab10')
FIGURES = PROJECT_ROOT / 'reports' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

config = load_config(PROJECT_ROOT / 'configs' / 'project.toml')
print('Ready.')

## 1. Load Raw Data

In [ ]:
raw_path = PROJECT_ROOT / 'data' / 'raw' / 'Ecommerce.csv'
df = normalize_columns(read_csv(raw_path))

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head(3)

## 2. Schema and Data Types

In [ ]:
schema = pd.DataFrame({
    'dtype'   : df.dtypes,
    'non_null': df.notna().sum(),
    'null_cnt': df.isna().sum(),
    'null_%'  : (df.isna().mean() * 100).round(2),
    'unique'  : df.nunique(),
})
print(schema.to_string())

## 3. Missing Values

In [ ]:
null_pct = (df.isna().mean() * 100).sort_values(ascending=False)
null_pct = null_pct[null_pct > 0]

if len(null_pct) > 0:
    fig, ax = plt.subplots(figsize=(8, max(2, len(null_pct) * 0.5)))
    null_pct.plot(kind='barh', ax=ax, color='steelblue')
    ax.set_xlabel('Missing (%)')
    ax.set_title('Missing Values by Column')
    for bar in ax.patches:
        ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
                f'{bar.get_width():.1f}%', va='center')
    plt.tight_layout()
    plt.savefig(FIGURES / '01_missing_values.png', dpi=120)
    plt.show()
    print(null_pct.to_string())
else:
    print('No missing values.')

## 4. Key Business Statistics

In [ ]:
# Detect column names
qty_col   = next((c for c in df.columns if 'quantity' in c), None)
price_col = next((c for c in df.columns if 'unit_price' in c or 'price' in c), None)
cust_col  = next((c for c in df.columns if 'customer' in c), None)
inv_col   = next((c for c in df.columns if 'invoice' in c and 'date' not in c), None)
date_col  = next((c for c in df.columns if 'date' in c), None)

print(f'Detected columns:')
for name, col in [('Customer', cust_col), ('Invoice', inv_col),
                   ('Date', date_col), ('Quantity', qty_col), ('Price', price_col)]:
    print(f'  {name:<12}: {col}')

In [ ]:
n_customers  = df[cust_col].nunique() if cust_col else 'N/A'
n_invoices   = df[inv_col].nunique() if inv_col else 'N/A'
n_cancelled  = df[inv_col].astype(str).str.startswith('C').sum() if inv_col else 0
n_no_cust    = df[cust_col].isna().sum() if cust_col else 0

if qty_col and price_col:
    df['gross_revenue'] = df[qty_col] * df[price_col]
    total_rev = df['gross_revenue'].sum()
    n_neg_qty = (df[qty_col] < 0).sum()
    n_bad_prc = (df[price_col] <= 0).sum()
else:
    total_rev = n_neg_qty = n_bad_prc = 'N/A'

print('=== Raw Dataset Summary ===')
print(f'Total rows           : {len(df):>12,}')
print(f'Unique customers     : {n_customers:>12,}')
print(f'Unique invoices      : {n_invoices:>12,}')
print(f'Rows w/o CustomerID  : {n_no_cust:>12,}  (will be dropped)')
print(f'Cancelled invoices   : {n_cancelled:>12,}  (will be dropped)')
print(f'Negative quantities  : {n_neg_qty:>12,}  (will be dropped)')
print(f'Zero/neg unit price  : {n_bad_prc:>12,}  (will be dropped)')
print(f'Total gross revenue  : £{total_rev:>11,.0f}')

## 5. Quantity and Price Distributions

Both features are **right-skewed** — a small number of orders have extremely high quantities or prices. This is expected in retail data and motivates the **log1p transformation** we will apply during feature engineering (see notebook 03).

In [ ]:
if qty_col and price_col:
    df_pos = df[(df[qty_col] > 0) & (df[price_col] > 0)].copy()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    q99_qty = df_pos[qty_col].quantile(0.99)
    axes[0].hist(df_pos[qty_col].clip(upper=q99_qty), bins=50,
                 color='steelblue', edgecolor='white', linewidth=0.5)
    axes[0].set_title('Quantity per Line (clipped at 99th pct)')
    axes[0].set_xlabel('Quantity')
    axes[0].set_ylabel('Frequency')

    q99_prc = df_pos[price_col].quantile(0.99)
    axes[1].hist(df_pos[price_col].clip(upper=q99_prc), bins=50,
                 color='darkorange', edgecolor='white', linewidth=0.5)
    axes[1].set_title('Unit Price per Line (clipped at 99th pct)')
    axes[1].set_xlabel('Unit Price (£)')

    plt.suptitle('Right-skewed distributions — log transform needed', fontsize=11, y=1.02)
    plt.tight_layout()
    plt.savefig(FIGURES / '01_raw_distributions.png', dpi=120)
    plt.show()

## 6. Transactions Over Time

In [ ]:
if date_col:
    df['date_parsed'] = pd.to_datetime(df[date_col], errors='coerce')
    monthly = df.dropna(subset=['date_parsed']).set_index('date_parsed').resample('ME').size()

    fig, ax = plt.subplots(figsize=(12, 4))
    monthly.plot(ax=ax, marker='o', linewidth=2, color='steelblue', markersize=5)
    ax.fill_between(monthly.index, monthly.values, alpha=0.15, color='steelblue')
    ax.set_title('Number of Transaction Lines per Month')
    ax.set_ylabel('Lines')
    ax.set_xlabel('')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    plt.tight_layout()
    plt.savefig(FIGURES / '01_transactions_over_time.png', dpi=120)
    plt.show()

    print(f'Date range: {df["date_parsed"].min().date()} to {df["date_parsed"].max().date()}')
    print(f'Note: November 2011 spike may indicate seasonal gift purchasing.')

## 7. Data Quality Summary

| Issue | Count | Action |
|---|---|---|
| Missing `CustomerID` | ~135,080 rows | **Drop** — cannot link to a customer |
| Cancelled invoices (`C` prefix) | ~9,288 rows | **Drop** — not a purchase |
| Negative quantity | ~10,624 rows | **Drop** — return or data error |
| Zero/negative unit price | ~2 rows | **Drop** — data error |

After cleaning, we expect approximately **370,000 valid rows** and **~4,300 unique customers**.

These steps are implemented in `src/insiders_loyalty_program/features.py → build_rfm_features()`.

In [ ]:
# Quick check — how many rows survive all filters?
if qty_col and price_col and cust_col and inv_col:
    mask = (
        df[cust_col].notna() &
        ~df[inv_col].astype(str).str.startswith('C', na=False) &
        (df[qty_col] > 0) &
        (df[price_col] > 0)
    )
    df_clean = df[mask]
    print(f'Clean rows  : {len(df_clean):,}')
    print(f'Dropped rows: {len(df) - len(df_clean):,}')
    print(f'Unique customers after cleaning: {df_clean[cust_col].nunique():,}')

---
**Next notebook:** `02_exploratory_analysis.ipynb` — deep EDA on customer behaviour and hypotheses.